In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_excel("../Hackathon/DataHackathon.xlsx")

In [ ]:
# Filtrar todas las variables que contienen 'Temp' o 'Temperature' en su nombre
temp_keywords = ["Temp", "Temperature", "PreformTemperatureControl"]
df_temp = df[df['variable'].str.contains('|'.join(temp_keywords), case=False, na=False)]

# Incluir también la variable de producción de botellas
df_prod = df[df['variable'] == "CONTIFORM_MMA.CONTIFORM_MMA1.WS_Tot_Bottles.0"]

# Renombrar la variable de producción de botellas
df_prod = df_prod.copy()
df_prod.loc[:, 'variable'] = "produccion_botellas"

# Unir ambas selecciones
df_filtered = pd.concat([df_temp, df_prod])

# Convertir la columna 'time' a formato datetime manejando formatos mixtos
df_filtered['time'] = pd.to_datetime(df_filtered['time'], errors='coerce', format='mixed')
df_filtered = df_filtered.dropna(subset=['time'])

# Convertir la columna 'value' a numérica, forzando errores a NaN
df_filtered['value'] = pd.to_numeric(df_filtered['value'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['value'])

# Verificar cuántas variables de temperatura se encontraron
print("Variables de temperatura seleccionadas:")
print(df_temp['variable'].unique())

# Filtrar solo variables de temperatura, excluyendo producción de botellas
df_temp_only = df_filtered[df_filtered['variable'] != "produccion_botellas"]

# Verificar si hay variables de temperatura para graficar
unique_variables = df_temp_only['variable'].unique()
num_vars = len(unique_variables)

In [ ]:
# Graficar las variables de temperatura en series de tiempo. Mostrar el comportamiento de su valor a través del tiempo
if num_vars == 0:
    print("No se encontraron variables de temperatura para graficar.")
else:
    fig, axes = plt.subplots(nrows=num_vars, ncols=1, figsize=(12, 3 * num_vars), sharex=True)

    if num_vars == 1:
        axes = [axes]

    for ax, variable in zip(axes, unique_variables):
        data = df_temp_only[df_temp_only['variable'] == variable]
        ax.plot(data['time'], data['value'], label=variable)
        ax.set_title(variable)
        ax.set_ylabel('Valor')
        ax.legend()
        ax.grid()

    plt.xlabel('Tiempo')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()